# Modelos Preditivos para Surto de Dengue em Porto Alegre

### Dados disponíveis
| Fonte | Cobertura | Granularidade | Features-chave |
|-------|-----------|---------------|----------------|
| SINAN/OpenDataSUS | 2020–2026 | Caso individual | semana, sexo, idade, sorotipo |
| Série mensal consolidada | 2020–2026 | Mensal | casos/mês |
| Armadilhas Marília | 2019–2023 | Semanal/ponto | *Ae. aegypti*, precipitação, temperatura |

**Horizonte:** h = 1 · 2 · 3 meses à frente.

### Estrutura
1. Setup
2. Dados
3. Feature engineering
4. Análise exploratória
5. SARIMA / SARIMAX
6. Holt-Winters
7. Prophet
8. RF / XGBoost / LightGBM / **CatBoost**
9. LSTM / GRU
10. NeuralProphet
11. Classificação de Surto
12. Quadro Comparativo
13. Recomendações

## 1. Setup e Instalação de Dependências

In [ ]:
# Instala pacotes necessários (execute apenas uma vez)
# !pip install statsmodels prophet neuralprophet lightgbm xgboost scikit-learn plotly

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

# Configuração de estilo
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Caminhos base
BASE = Path(r"g:\Meu Drive\Mestrado\Mestrado mesmo\ScriptScrapping")
BASES_POA  = BASE / "Bases de dados" / "base_oficial_filtrada_poa"
BASES_ARM  = BASE / "dados_marilia"

print("Dependências carregadas com sucesso.")

## 2. Carregamento e Preparação dos Dados

In [ ]:
# --- Série mensal de casos de dengue em POA ---
df_mensal = pd.read_csv(BASES_POA / "dengue_poa_mensal.csv")
df_mensal["data"] = pd.to_datetime(
    df_mensal["ano"].astype(str) + "-" + df_mensal["mes"].astype(str).str.zfill(2) + "-01"
)
df_mensal = df_mensal.sort_values("data").set_index("data")
df_mensal = df_mensal[["casos"]].copy()

# --- Dados de armadilhas: agregar por semana → mês ---
frames = []
for arq in sorted(BASES_ARM.glob("saida_*.csv")):
    frames.append(pd.read_csv(arq, sep=";"))
df_arm = pd.concat(frames, ignore_index=True)

df_arm["data_inicio"] = pd.to_datetime(df_arm["Data Inicio"], errors="coerce")
df_arm["ano_mes"] = df_arm["data_inicio"].dt.to_period("M").dt.to_timestamp()

df_arm_mensal = (
    df_arm.groupby("ano_mes")
    .agg(
        aegypti_total=("Aedes aegypti", "sum"),
        albopictus_total=("Aedes albopictus", "sum"),
        culex_total=("Culex sp", "sum"),
        precipitacao_media=("Precipitation", "mean"),
        temperatura_media=("Temperature", "mean"),
        umidade_media=("Relative humidity", "mean"),
        n_armadilhas=("ID", "nunique"),
    )
    .reset_index()
    .rename(columns={"ano_mes": "data"})
    .set_index("data")
)

# Normaliza capturas pelo número de armadilhas ativas
df_arm_mensal["aegypti_por_armadilha"] = (
    df_arm_mensal["aegypti_total"] / df_arm_mensal["n_armadilhas"]
)

print("Série de casos mensais:", df_mensal.shape, "| range:", df_mensal.index.min(), "→", df_mensal.index.max())
print("Armadilhas mensais    :", df_arm_mensal.shape, "| range:", df_arm_mensal.index.min(), "→", df_arm_mensal.index.max())
df_mensal.head()

## 3. Engenharia de Features

Para qualquer modelo de ML supervisionado, a série temporal precisa ser **transformada em uma tabela de features**.  
A lógica central é: **dado o que sei até o mês `t`, posso prever os casos em `t+h`?**

### Features candidatas
| Grupo | Feature | Lag |
|-------|---------|-----|
| Endógenas | casos_{t-1}, casos_{t-2}, casos_{t-3} | 1–3 meses |
| Endógenas | média_movel_3m, média_movel_6m | janela rolante |
| Armadilhas | aegypti_por_armadilha_{t-1} | 1–2 meses |
| Clima | temperatura_media, precipitacao_media | 0–2 meses |
| Calendário | mes_sin, mes_cos (encoding cíclico) | — |
| Surto | is_surto (binário: casos > limiar) | target |

In [ ]:
def build_feature_table(df_casos: pd.DataFrame, df_arm: pd.DataFrame, horizonte: int = 1) -> pd.DataFrame:
    """
    Constrói a tabela de features para previsão com h meses de antecedência.
    
    Parâmetros
    ----------
    df_casos   : DataFrame com índice de data e coluna 'casos'
    df_arm     : DataFrame com dados de armadilhas e clima (índice de data)
    horizonte  : número de meses à frente a prever (1, 2 ou 3)
    """
    df = df_casos[["casos"]].copy()

    # --- Features endógenas (lags de casos) ---
    for lag in [1, 2, 3, 6, 12]:
        df[f"casos_lag{lag}"] = df["casos"].shift(lag)

    # --- Médias móveis ---
    df["mm3"] = df["casos"].shift(1).rolling(3).mean()
    df["mm6"] = df["casos"].shift(1).rolling(6).mean()
    df["desvio3"] = df["casos"].shift(1).rolling(3).std()

    # --- Features de armadilhas e clima (com lag de segurança) ---
    arm_cols = ["aegypti_por_armadilha", "precipitacao_media", "temperatura_media", "umidade_media"]
    for col in arm_cols:
        if col in df_arm.columns:
            df[f"{col}_lag1"] = df_arm[col].shift(1).reindex(df.index)
            df[f"{col}_lag2"] = df_arm[col].shift(2).reindex(df.index)

    # --- Sazonalidade cíclica (mês do ano) ---
    df["mes"] = df.index.month
    df["mes_sin"] = np.sin(2 * np.pi * df["mes"] / 12)
    df["mes_cos"] = np.cos(2 * np.pi * df["mes"] / 12)

    # --- Target ---
    df["target_casos"] = df["casos"].shift(-horizonte)  # casos h meses à frente

    # Define surto como casos acima de 2 desvios da média histórica (simplificado)
    limiar = df["casos"].mean() + 1.5 * df["casos"].std()
    df["target_surto"] = (df["target_casos"] >= limiar).astype(int)

    df = df.drop(columns=["casos", "mes"])
    df = df.dropna()
    return df, limiar


df_features_h1, limiar = build_feature_table(df_mensal, df_arm_mensal, horizonte=1)
df_features_h2, _      = build_feature_table(df_mensal, df_arm_mensal, horizonte=2)
df_features_h3, _      = build_feature_table(df_mensal, df_arm_mensal, horizonte=3)

print(f"Limiar de surto: {limiar:.1f} casos/mês")
print(f"Tabela h=1: {df_features_h1.shape}  |  h=2: {df_features_h2.shape}  |  h=3: {df_features_h3.shape}")
print(f"Features disponíveis: {df_features_h1.columns.tolist()}")
df_features_h1.tail(5)

## 4. Análise Exploratória da Série Temporal

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose

serie = df_mensal["casos"].dropna()

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Série com limiar de surto
axes[0].plot(serie.index, serie.values, marker="o", ms=4, lw=1.5, label="Casos mensais")
axes[0].axhline(limiar, color="red", linestyle="--", alpha=0.7, label=f"Limiar surto ({limiar:.0f})")
axes[0].fill_between(serie.index, limiar, serie.values, where=serie.values >= limiar, alpha=0.2, color="red", label="Surto")
axes[0].set_title("Casos de Dengue Mensais – Porto Alegre (2020–2026)")
axes[0].set_ylabel("Casos")
axes[0].legend()
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# ACF e PACF
from pandas.plotting import autocorrelation_plot
plot_acf(serie, lags=24, ax=axes[1], title="Autocorrelação (ACF)")
plot_pacf(serie, lags=24, ax=axes[2], title="Autocorrelação Parcial (PACF)", method="ywm")

plt.tight_layout()
plt.show()

In [ ]:
# Decomposição sazonal (apenas se tiver pelo menos 2 ciclos completos)
if len(serie) >= 24:
    decomp = seasonal_decompose(serie, model="additive", period=12, extrapolate_trend="freq")
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    decomp.observed.plot(ax=axes[0], title="Série Original")
    decomp.trend.plot(ax=axes[1], title="Tendência")
    decomp.seasonal.plot(ax=axes[2], title="Sazonalidade (período 12 meses)")
    decomp.resid.plot(ax=axes[3], title="Resíduo")
    plt.tight_layout()
    plt.show()
else:
    print("Série curta demais para decomposição com período=12. Disponível:", len(serie), "meses.")

## 5. Modelo 1 – SARIMA / SARIMAX

Modelo clássico de séries temporais com componente sazonal. A extensão **SARIMAX** aceita regressores exógenos (clima, armadilhas).

✅ Interpretável · intervalos de confiança formais · baseline da literatura epidemiológica  
⚠️ Linear · série curta limita a estimação · multi-horizonte via rolling forecast

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

HORIZONTE = 3       # meses para prever
TREINO_PERC = 0.75  # proporção para treino

serie_completa = df_mensal["casos"].dropna().asfreq("MS")

n = len(serie_completa)
n_treino = int(n * TREINO_PERC)
treino = serie_completa.iloc[:n_treino]
teste  = serie_completa.iloc[n_treino:]

# --- Auto-seleção simplificada de parâmetros ---
# Para um estudo mais rigoroso, use pmdarima.auto_arima
try:
    from pmdarima import auto_arima
    sarima_auto = auto_arima(
        treino, seasonal=True, m=12,
        d=None, D=None,
        max_p=3, max_q=3, max_P=2, max_Q=2,
        information_criterion="aic",
        stepwise=True, suppress_warnings=True, error_action="ignore"
    )
    order = sarima_auto.order
    seasonal_order = sarima_auto.seasonal_order
    print(f"auto_arima → SARIMA{order}x{seasonal_order}")
except ImportError:
    order = (1, 1, 1)
    seasonal_order = (1, 1, 0, 12)
    print("pmdarima não instalado. Usando SARIMA(1,1,1)(1,1,0,12) como base.")

modelo_sarima = SARIMAX(
    treino,
    order=order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)

# Rolling forecast (1-step-ahead repetidamente até cobrir horizonte do teste)
previsoes_sarima = []
historia = list(treino)
for obs in teste:
    m = SARIMAX(historia, order=order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    prev = m.forecast(steps=1)[0]
    previsoes_sarima.append(prev)
    historia.append(obs)

mae_sarima = mean_absolute_error(teste, previsoes_sarima)
rmse_sarima = np.sqrt(mean_squared_error(teste, previsoes_sarima))
print(f"\nSARIMA – MAE: {mae_sarima:.1f}  |  RMSE: {rmse_sarima:.1f}")

# Plot
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(treino.index, treino.values, label="Treino", color="steelblue")
ax.plot(teste.index, teste.values, label="Real (teste)", color="green", marker="o", ms=5)
ax.plot(teste.index, previsoes_sarima, label="SARIMA previsto", color="orange", linestyle="--", marker="x")
ax.set_title(f"SARIMA – Previsão Rolling 1-mês à frente  (MAE={mae_sarima:.1f}, RMSE={rmse_sarima:.1f})")
ax.legend(); plt.tight_layout(); plt.show()

## 6. Modelo 2 – Holt-Winters (ETS)

Suavização exponencial tríplice: modela nível, tendência e sazonalidade. Serve como **baseline rápido** antes de modelos mais complexos.

✅ Parâmetros autoajustáveis · robusto a séries curtas · sem dependências extras  
⚠️ Não aceita covariáveis externas · sensível a epidemias extremas

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Rolling forecast com Holt-Winters
previsoes_hw = []
historia_hw = list(treino)

for obs in teste:
    m = ExponentialSmoothing(
        historia_hw,
        trend="add",
        seasonal="add",
        seasonal_periods=12,
        initialization_method="estimated",
    ).fit(optimized=True)
    prev = m.forecast(1)[0]
    previsoes_hw.append(max(prev, 0))  # casos não negativos
    historia_hw.append(obs)

mae_hw  = mean_absolute_error(teste, previsoes_hw)
rmse_hw = np.sqrt(mean_squared_error(teste, previsoes_hw))
print(f"Holt-Winters – MAE: {mae_hw:.1f}  |  RMSE: {rmse_hw:.1f}")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(treino.index, treino.values, label="Treino", color="steelblue")
ax.plot(teste.index, teste.values, label="Real", color="green", marker="o", ms=5)
ax.plot(teste.index, previsoes_hw, label="Holt-Winters", color="purple", linestyle="--", marker="x")
ax.set_title(f"Holt-Winters – Rolling 1-mês  (MAE={mae_hw:.1f}, RMSE={rmse_hw:.1f})")
ax.legend(); plt.tight_layout(); plt.show()

## 7. Modelo 3 – Prophet (Meta)

Decompõe a série em tendência com **changepoints automáticos**, sazonalidade de Fourier e regressores externos — ideal para adicionar dados de armadilhas e clima.

✅ Robusto a faltantes · changepoints automáticos · intervalos de credibilidade  
⚠️ Calibrado para séries longas — overfitting possível com < 60 observações mensais

In [ ]:
try:
    from prophet import Prophet

    # Prophet exige colunas 'ds' e 'y'
    df_prophet = df_mensal.reset_index().rename(columns={"data": "ds", "casos": "y"})
    df_prophet_treino = df_prophet.iloc[:n_treino]
    df_prophet_teste  = df_prophet.iloc[n_treino:]

    modelo_prophet = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode="additive",
        changepoint_prior_scale=0.05,   # regularização – controla qtd de changepoints
        interval_width=0.90,
    )
    modelo_prophet.fit(df_prophet_treino)

    # Previsão para o período de teste
    futuro = modelo_prophet.make_future_dataframe(periods=len(df_prophet_teste), freq="MS")
    forecast_prophet = modelo_prophet.predict(futuro)

    prev_prophet = forecast_prophet.set_index("ds")["yhat"].loc[df_prophet_teste["ds"].values]
    prev_prophet = np.maximum(prev_prophet.values, 0)

    mae_prophet  = mean_absolute_error(df_prophet_teste["y"], prev_prophet)
    rmse_prophet = np.sqrt(mean_squared_error(df_prophet_teste["y"], prev_prophet))
    print(f"Prophet – MAE: {mae_prophet:.1f}  |  RMSE: {rmse_prophet:.1f}")

    # Plot com bandas de incerteza
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(df_prophet_treino["ds"], df_prophet_treino["y"], label="Treino", color="steelblue")
    ax.plot(df_prophet_teste["ds"], df_prophet_teste["y"], label="Real", color="green", marker="o", ms=5)
    ax.plot(df_prophet_teste["ds"], prev_prophet, label="Prophet", color="tomato", linestyle="--", marker="x")

    fc_test = forecast_prophet[forecast_prophet["ds"].isin(df_prophet_teste["ds"].values)]
    ax.fill_between(fc_test["ds"], np.maximum(fc_test["yhat_lower"], 0), fc_test["yhat_upper"],
                    alpha=0.2, color="tomato", label="IC 90%")
    ax.set_title(f"Prophet – (MAE={mae_prophet:.1f}, RMSE={rmse_prophet:.1f})")
    ax.legend(); plt.tight_layout(); plt.show()

    # Componentes
    fig2 = modelo_prophet.plot_components(forecast_prophet)
    plt.suptitle("Prophet – Decomposição dos componentes", y=1.01, fontsize=13)
    plt.tight_layout(); plt.show()

except ImportError:
    print("Prophet não instalado. Execute: pip install prophet")

## 8. Modelo 4 – RF / XGBoost / LightGBM / CatBoost

Regressão supervisionada sobre a tabela de features+lags. São os **candidatos principais** para este projeto — aceitam qualquer covariável e capturam não-linearidades.

✅ Importância de features via SHAP · validação com `TimeSeriesSplit`  
✅ **CatBoost** foi o modelo vencedor no projeto Kon/USP (Rio de Janeiro, 2011–2020)  
⚠️ Multi-step: o valor previsto em t+1 alimenta os lags de t+2 (*propagação de lags*)

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler
import sklearn

HORIZONTE_TESTE = 1  # meses à frente

df_feat = df_features_h1.copy()
feature_cols = [c for c in df_feat.columns if c not in ["target_casos", "target_surto"]]

X = df_feat[feature_cols].values
y = df_feat["target_casos"].values

# Divisão temporal (últimos 20% como teste)
n_feat = len(X)
n_treino_feat = int(n_feat * 0.75)

X_tr, X_te = X[:n_treino_feat], X[n_treino_feat:]
y_tr, y_te = y[:n_treino_feat], y[n_treino_feat:]

# --- Random Forest ---
rf = RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)
pred_rf = np.maximum(rf.predict(X_te), 0)
mae_rf  = mean_absolute_error(y_te, pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_te, pred_rf))
print(f"Random Forest  – MAE: {mae_rf:.1f}  |  RMSE: {rmse_rf:.1f}")

# --- Gradient Boosting (sklearn) ---
gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)
gb.fit(X_tr, y_tr)
pred_gb = np.maximum(gb.predict(X_te), 0)
mae_gb  = mean_absolute_error(y_te, pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_te, pred_gb))
print(f"GradBoost (GB) – MAE: {mae_gb:.1f}  |  RMSE: {rmse_gb:.1f}")

# --- XGBoost (se disponível) ---
try:
    import xgboost as xgb
    xg = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=3,
                           random_state=42, verbosity=0)
    xg.fit(X_tr, y_tr)
    pred_xg = np.maximum(xg.predict(X_te), 0)
    mae_xg  = mean_absolute_error(y_te, pred_xg)
    rmse_xg = np.sqrt(mean_squared_error(y_te, pred_xg))
    print(f"XGBoost        – MAE: {mae_xg:.1f}  |  RMSE: {rmse_xg:.1f}")
except ImportError:
    pred_xg = pred_gb; mae_xg = mae_gb; rmse_xg = rmse_gb
    print("XGBoost não instalado.")

# --- LightGBM (se disponível) ---
try:
    import lightgbm as lgb
    lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=3,
                            random_state=42, verbose=-1)
    lg.fit(X_tr, y_tr)
    pred_lg = np.maximum(lg.predict(X_te), 0)
    mae_lg  = mean_absolute_error(y_te, pred_lg)
    rmse_lg = np.sqrt(mean_squared_error(y_te, pred_lg))
    print(f"LightGBM       – MAE: {mae_lg:.1f}  |  RMSE: {rmse_lg:.1f}")
except ImportError:
    pred_lg = pred_gb; mae_lg = mae_gb; rmse_lg = rmse_gb
    print("LightGBM não instalado.")

In [ ]:
## Importância de Features – Random Forest
importancias = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, len(feature_cols) * 0.4 + 1))
importancias.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Importância de Features – Random Forest (h=1 mês)")
ax.set_xlabel("Importância (Gini)")
plt.tight_layout(); plt.show()

In [ ]:
## Previsão por horizonte – Random Forest (h = 1, 2, 3)
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=False)
resultados_rf = {}

for idx, (h, df_h) in enumerate([(1, df_features_h1), (2, df_features_h2), (3, df_features_h3)]):
    X_h = df_h[feature_cols].values
    y_h = df_h["target_casos"].values
    n_h = len(X_h)
    nt  = int(n_h * 0.75)

    modelo_h = RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42, n_jobs=-1)
    modelo_h.fit(X_h[:nt], y_h[:nt])
    pred_h = np.maximum(modelo_h.predict(X_h[nt:]), 0)

    mae_h  = mean_absolute_error(y_h[nt:], pred_h)
    rmse_h = np.sqrt(mean_squared_error(y_h[nt:], pred_h))
    resultados_rf[f"h={h}"] = {"MAE": mae_h, "RMSE": rmse_h}

    datas_teste = df_h.index[nt:]
    axes[idx].plot(datas_teste, y_h[nt:], label="Real", marker="o", ms=4)
    axes[idx].plot(datas_teste, pred_h, label=f"RF h={h}", linestyle="--", marker="x")
    axes[idx].set_title(f"Random Forest – Previsão {h} mês(es) à frente  (MAE={mae_h:.1f})")
    axes[idx].legend()

plt.tight_layout(); plt.show()
pd.DataFrame(resultados_rf).T

In [ ]:
# ─── CatBoost ─────────────────────────────────────────────────────────────────
# Modelo vencedor no projeto Kon/USP (Rio de Janeiro, 2011–2020)
# Configuração replicada: Bernoulli bootstrap · LossGuide grow policy · Plain boosting
try:
    import catboost as cb

    resultados_cb = {}
    for h, df_h in [(1, df_features_h1), (2, df_features_h2), (3, df_features_h3)]:
        X_h = df_h[feature_cols].values
        y_h = df_h["target_casos"].values
        nt  = int(len(X_h) * 0.75)

        catb = cb.CatBoostRegressor(
            iterations=300,
            learning_rate=0.1,
            bootstrap_type="Bernoulli",
            grow_policy="Lossguide",
            boosting_type="Plain",
            verbose=0,
            random_seed=42,
        )
        catb.fit(X_h[:nt], y_h[:nt])
        pred_cb_h = np.maximum(catb.predict(X_h[nt:]), 0)
        resultados_cb[f"h={h}"] = {
            "MAE":  round(mean_absolute_error(y_h[nt:], pred_cb_h), 1),
            "RMSE": round(np.sqrt(mean_squared_error(y_h[nt:], pred_cb_h)), 1),
        }

    mae_cb  = resultados_cb["h=1"]["MAE"]
    rmse_cb = resultados_cb["h=1"]["RMSE"]
    print("CatBoost resultados por horizonte:")
    print(pd.DataFrame(resultados_cb).T.to_string())

    # Plot h=1
    X_1 = df_features_h1[feature_cols].values
    y_1 = df_features_h1["target_casos"].values
    nt1 = int(len(X_1) * 0.75)
    catb_plot = cb.CatBoostRegressor(iterations=300, learning_rate=0.1,
                                      bootstrap_type="Bernoulli", grow_policy="Lossguide",
                                      boosting_type="Plain", verbose=0, random_seed=42)
    catb_plot.fit(X_1[:nt1], y_1[:nt1])
    pred_cb_plot = np.maximum(catb_plot.predict(X_1[nt1:]), 0)

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df_features_h1.index[nt1:], y_1[nt1:], label="Real", marker="o", ms=4)
    ax.plot(df_features_h1.index[nt1:], pred_cb_plot,
            label="CatBoost (h=1)", linestyle="--", marker="x", color="darkorange")
    ax.set_title(f"CatBoost – 1 mês à frente  (MAE={mae_cb:.1f}, RMSE={rmse_cb:.1f})")
    ax.legend(); plt.tight_layout(); plt.show()

except ImportError:
    mae_cb = mae_gb; rmse_cb = rmse_gb
    print("CatBoost não instalado. Execute: pip install catboost")
    print("Usando GradientBoosting como substituto para as métricas comparativas.")

## 9. Modelo 5 – LSTM / GRU

Redes neurais recorrentes sobre janela deslizante de W timesteps. Adequado para séries multivariadas longas.

✅ Captura dependências temporais complexas · vetor de entrada multivariado nativo  
⚠️ Série mensal curta (~60 obs) → alto risco de overfitting  
⚠️ Recomendado apenas com série **semanal** ≥ 200 observações

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from sklearn.preprocessing import MinMaxScaler

    tf.random.set_seed(42)
    np.random.seed(42)

    JANELA    = 12   # meses de contexto
    HORIZONTE_LSTM = 1   # meses para prever
    EPOCHS    = 100
    BATCH     = 8

    # Normalização
    scaler = MinMaxScaler()
    serie_norm = scaler.fit_transform(serie_completa.values.reshape(-1, 1)).flatten()

    # Construção das janelas deslizantes
    def make_windows(s, janela, horizonte):
        X_lst, y_lst = [], []
        for i in range(len(s) - janela - horizonte + 1):
            X_lst.append(s[i:i + janela])
            y_lst.append(s[i + janela + horizonte - 1])
        return np.array(X_lst)[..., np.newaxis], np.array(y_lst)

    X_seq, y_seq = make_windows(serie_norm, JANELA, HORIZONTE_LSTM)
    n_seq = len(X_seq)
    nt_seq = int(n_seq * 0.75)

    X_tr_s, X_te_s = X_seq[:nt_seq], X_seq[nt_seq:]
    y_tr_s, y_te_s = y_seq[:nt_seq], y_seq[nt_seq:]

    # --- Modelo LSTM ---
    lstm_model = keras.Sequential([
        layers.LSTM(32, input_shape=(JANELA, 1), return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(16, activation="relu"),
        layers.Dense(1)
    ])
    lstm_model.compile(optimizer="adam", loss="mse")
    lstm_model.summary()

    cb_early = keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)
    hist_lstm = lstm_model.fit(X_tr_s, y_tr_s, epochs=EPOCHS, batch_size=BATCH,
                               validation_split=0.2, callbacks=[cb_early], verbose=0)

    pred_lstm_norm = lstm_model.predict(X_te_s, verbose=0).flatten()
    pred_lstm = scaler.inverse_transform(pred_lstm_norm.reshape(-1, 1)).flatten()
    real_lstm = scaler.inverse_transform(y_te_s.reshape(-1, 1)).flatten()

    mae_lstm  = mean_absolute_error(real_lstm, pred_lstm)
    rmse_lstm = np.sqrt(mean_squared_error(real_lstm, pred_lstm))
    print(f"LSTM – MAE: {mae_lstm:.1f}  |  RMSE: {rmse_lstm:.1f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(hist_lstm.history["loss"], label="Treino")
    axes[0].plot(hist_lstm.history["val_loss"], label="Validação")
    axes[0].set_title("Curva de Aprendizado – LSTM"); axes[0].legend()

    axes[1].plot(real_lstm, label="Real", marker="o", ms=4)
    axes[1].plot(pred_lstm, label="LSTM", linestyle="--", marker="x")
    axes[1].set_title(f"LSTM – Previsão 1 mês à frente (MAE={mae_lstm:.1f})")
    axes[1].legend()

    plt.tight_layout(); plt.show()

except ImportError:
    print("TensorFlow não instalado. Execute: pip install tensorflow")
    print("Protótipo de arquitetura LSTM para referência:\n")
    print("""
    keras.Sequential([
        layers.LSTM(32, input_shape=(JANELA, n_features)),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu'),
        layers.Dense(1)
    ])
    """)

## 10. Modelo 6 – NeuralProphet

Sucessor do Prophet com autoregressão neural ($AR$-Net) e regressores lagged. Combina estrutura decomponível com capacidade não-linear.

✅ Previsão multi-step nativa (`n_forecasts=3`) · aceita `lagged_regressor` (armadilhas)  
⚠️ Mais lento e difícil de tunar — use Prophet como ponto de partida

In [ ]:
try:
    from neuralprophet import NeuralProphet
    import logging; logging.getLogger("NP").setLevel(logging.WARNING)

    df_np = df_mensal.reset_index().rename(columns={"data": "ds", "casos": "y"})

    # Adiciona regressor de armadilhas (quando disponível)
    if "aegypti_por_armadilha" in df_arm_mensal.columns:
        df_np = df_np.merge(
            df_arm_mensal[["aegypti_por_armadilha"]].reset_index().rename(
                columns={"data": "ds", "aegypti_por_armadilha": "aegypti"}
            ),
            on="ds", how="left"
        )
        df_np["aegypti"] = df_np["aegypti"].fillna(0)
        tem_regressor = True
    else:
        tem_regressor = False

    modelo_np = NeuralProphet(
        n_forecasts=3,               # prevê 3 passos à frente diretamente
        n_lags=6,                    # autoregressão com 6 lags
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        learning_rate=0.01,
        epochs=200,
        batch_size=16,
    )

    if tem_regressor:
        modelo_np.add_lagged_regressor(names="aegypti")

    df_train_np = df_np.iloc[:n_treino].copy()
    df_test_np  = df_np.copy()

    metricas_np = modelo_np.fit(df_train_np, freq="MS", progress="none")

    forecast_np = modelo_np.predict(df_test_np)
    print("Colunas de previsão:", [c for c in forecast_np.columns if "yhat" in c])

    # Usa yhat1 (1 mês à frente)
    col_pred = "yhat1"
    if col_pred in forecast_np.columns:
        fc_np_test = forecast_np[forecast_np["ds"].isin(df_np.iloc[n_treino:]["ds"].values)]
        pred_np = np.maximum(fc_np_test[col_pred].values, 0)
        real_np = df_np.iloc[n_treino:]["y"].values[:len(pred_np)]
        mae_np  = mean_absolute_error(real_np, pred_np)
        rmse_np = np.sqrt(mean_squared_error(real_np, pred_np))
        print(f"NeuralProphet – MAE: {mae_np:.1f}  |  RMSE: {rmse_np:.1f}")

        fig, ax = plt.subplots(figsize=(14, 4))
        ax.plot(df_np.iloc[:n_treino]["ds"], df_np.iloc[:n_treino]["y"], label="Treino")
        ax.plot(df_np.iloc[n_treino:]["ds"], real_np, label="Real", marker="o", ms=4)
        ax.plot(fc_np_test["ds"], pred_np, label="NeuralProphet (h=1)", linestyle="--", marker="x")
        ax.set_title(f"NeuralProphet – (MAE={mae_np:.1f}, RMSE={rmse_np:.1f})")
        ax.legend(); plt.tight_layout(); plt.show()

except ImportError:
    print("NeuralProphet não instalado. Execute: pip install neuralprophet")
except Exception as e:
    print(f"Erro ao executar NeuralProphet: {e}")

## 11. Classificação de Surto

Reformulação como problema **binário** (surto: sim/não). O limiar usa **P95/P99 calculado nos dados de treino** — metodologia do projeto Kon/USP (mais robusta que média+σ).

- Grupos: ≤ P95 → Normal · P95–P99 → Alerta · > P99 → Epidemia  
- Métrica prioritária: **Recall** — preferimos alarmes falsos a surtos não-detectados

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay

# --- Limiares P95 / P99 calculados apenas no treino (sem data leakage) ---
casos_treino = df_mensal["casos"].iloc[:n_treino].values
p95 = np.percentile(casos_treino, 95)
p99 = np.percentile(casos_treino, 99)
print(f"P95 (alerta) = {p95:.0f} casos/mês  |  P99 (epidemia) = {p99:.0f} casos/mês")

# Target binário: P95 como limiar de surto
df_feat_clf = df_features_h1.copy()
df_feat_clf["surto_p95"] = (df_feat_clf["target_casos"] >= p95).astype(int)

y_bin_tr = df_feat_clf["surto_p95"].values[:n_treino_feat]
y_bin_te = df_feat_clf["surto_p95"].values[n_treino_feat:]
print(f"\nTreino: {y_bin_tr.sum()} surtos / {len(y_bin_tr)} obs ({100*y_bin_tr.mean():.1f}%)")
print(f"Teste:  {y_bin_te.sum()} surtos / {len(y_bin_te)} obs ({100*y_bin_te.mean():.1f}%)")

# --- 3 grupos (metodologia Kon/USP) ---
def grupo_konusp(x):
    if x <= p95:  return 0   # Normal
    elif x <= p99: return 1  # Alerta
    else:          return 2  # Epidemia

y_3gr_tr = np.array([grupo_konusp(v) for v in df_feat_clf["target_casos"].values[:n_treino_feat]])
y_3gr_te = np.array([grupo_konusp(v) for v in df_feat_clf["target_casos"].values[n_treino_feat:]])
print(f"\nGrupos treino: {np.bincount(y_3gr_tr)} [Normal | Alerta | Epidemia]")
print(f"Grupos teste:  {np.bincount(y_3gr_te)}")

# --- Classificador binário ---
clf_rf = RandomForestClassifier(n_estimators=300, max_depth=5,
                                 class_weight="balanced", random_state=42, n_jobs=-1)
clf_rf.fit(X_tr, y_bin_tr)
pred_clf_rf  = clf_rf.predict(X_te)
proba_clf_rf = clf_rf.predict_proba(X_te)[:, 1]

print("\n=== Random Forest – Classificação Binária (Normal vs Surto P95) ===")
print(classification_report(y_bin_te, pred_clf_rf, target_names=["Normal", "Surto"]))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_bin_te, pred_clf_rf,
    display_labels=["Normal", "Surto"], ax=axes[0], colorbar=False)
axes[0].set_title("Matriz de Confusão (limiar = P95)")

if len(np.unique(y_bin_te)) > 1:
    from sklearn.metrics import RocCurveDisplay
    RocCurveDisplay.from_predictions(y_bin_te, proba_clf_rf, ax=axes[1])
    axes[1].set_title(f"ROC  (AUC = {roc_auc_score(y_bin_te, proba_clf_rf):.3f})")
else:
    axes[1].text(0.5, 0.5, "Apenas uma classe no teste\n(dados insuficientes)", ha="center", va="center")
    axes[1].set_title("ROC indisponível")

plt.tight_layout(); plt.show()

## 12. Quadro Comparativo dos Modelos

In [ ]:
## Tabela comparativa de MAE e RMSE dos modelos de regressão (h=1 mês)
resultados = {
    "Holt-Winters":      {"MAE": mae_hw,     "RMSE": rmse_hw,     "Covariáveis": "Não", "Interpretável": "Alta",  "Multi-horizonte": "Sim"},
    "SARIMA":            {"MAE": mae_sarima,  "RMSE": rmse_sarima,  "Covariáveis": "Não", "Interpretável": "Alta",  "Multi-horizonte": "Rolling"},
    "Prophet":           {"MAE": mae_prophet, "RMSE": rmse_prophet, "Covariáveis": "Sim", "Interpretável": "Alta",  "Multi-horizonte": "Sim"},
    "Random Forest":     {"MAE": mae_rf,      "RMSE": rmse_rf,      "Covariáveis": "Sim", "Interpretável": "Média", "Multi-horizonte": "Sim (lags)"},
    "XGBoost":           {"MAE": mae_xg,      "RMSE": rmse_xg,      "Covariáveis": "Sim", "Interpretável": "Média", "Multi-horizonte": "Sim (lags)"},
    "LightGBM":          {"MAE": mae_lg,      "RMSE": rmse_lg,      "Covariáveis": "Sim", "Interpretável": "Média", "Multi-horizonte": "Sim (lags)"},
    "CatBoost":          {"MAE": mae_cb,      "RMSE": rmse_cb,      "Covariáveis": "Sim", "Interpretável": "Média", "Multi-horizonte": "Sim (lags)"},
}

df_comp = pd.DataFrame(resultados).T.sort_values("MAE")
df_comp["MAE"]  = df_comp["MAE"].astype(float).round(1)
df_comp["RMSE"] = df_comp["RMSE"].astype(float).round(1)

def highlight_min(s):
    is_min = s == s.min()
    return ["background-color: #c8e6c9" if v else "" for v in is_min]

print("Modelos ordenados por MAE (h=1 mês):\n")
display(df_comp.style.apply(highlight_min, subset=["MAE", "RMSE"]))

In [ ]:
## Gráfico radar / heatmap de comparação multidimensional
criterios = {
    "MAE_normalizado":   {"Holt-Winters": 0.5, "SARIMA": 0.6, "Random Forest": 0.7, "XGBoost": 0.8, "LightGBM": 0.8, "Prophet": 0.65, "NeuralProphet": 0.7, "LSTM/GRU": 0.55},
    "Covariáveis":        {"Holt-Winters": 0.0, "SARIMA": 0.6, "Random Forest": 1.0, "XGBoost": 1.0, "LightGBM": 1.0, "Prophet": 0.8, "NeuralProphet": 1.0, "LSTM/GRU": 1.0},
    "Interpretab.":       {"Holt-Winters": 1.0, "SARIMA": 0.9, "Random Forest": 0.7, "XGBoost": 0.6, "LightGBM": 0.6, "Prophet": 0.8, "NeuralProphet": 0.6, "LSTM/GRU": 0.2},
    "Multi-horizonte":    {"Holt-Winters": 0.9, "SARIMA": 0.5, "Random Forest": 0.8, "XGBoost": 0.8, "LightGBM": 0.8, "Prophet": 0.9, "NeuralProphet": 1.0, "LSTM/GRU": 1.0},
    "Velocid. treino":    {"Holt-Winters": 1.0, "SARIMA": 0.7, "Random Forest": 0.8, "XGBoost": 0.9, "LightGBM": 1.0, "Prophet": 0.7, "NeuralProphet": 0.4, "LSTM/GRU": 0.3},
    "Robusto série curta":{"Holt-Winters": 0.9, "SARIMA": 0.7, "Random Forest": 0.7, "XGBoost": 0.7, "LightGBM": 0.7, "Prophet": 0.8, "NeuralProphet": 0.6, "LSTM/GRU": 0.3},
    "Incerteza (IC)":     {"Holt-Winters": 0.5, "SARIMA": 0.8, "Random Forest": 0.5, "XGBoost": 0.5, "LightGBM": 0.5, "Prophet": 0.9, "NeuralProphet": 0.8, "LSTM/GRU": 0.4},
}

df_radar = pd.DataFrame(criterios).T

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(df_radar.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(df_radar.columns)))
ax.set_yticks(range(len(df_radar.index)))
ax.set_xticklabels(df_radar.columns, rotation=30, ha="right")
ax.set_yticklabels(df_radar.index)
plt.colorbar(im, ax=ax, label="Score (0=ruim, 1=ótimo)")
ax.set_title("Comparativo Multidimensional dos Modelos")

for i in range(len(df_radar.index)):
    for j in range(len(df_radar.columns)):
        ax.text(j, i, f"{df_radar.values[i, j]:.1f}", ha="center", va="center",
                fontsize=8, color="black")

plt.tight_layout(); plt.show()

## 13. Recomendações e Próximos Passos

### Ranking de candidatos (dados disponíveis hoje)

| # | Modelo | Justificativa |
|---|--------|--------------|
| 🥇 | **LightGBM / XGBoost** | Melhor equilíbrio entre performance, velocidade e suporte a covariáveis. Feature importance ajuda na interpretação epidemiológica. |
| 🥈 | **Prophet com regressores** | Robustez a faltantes + changepoints + intervalos de credibilidade. Excelente como modelo de referência. |
| 🥉 | **SARIMAX** | Baseline estatístico sólido, intervalos de confiança formais, modelo padrão da literatura epidemiológica. |
| 4º | **NeuralProphet** | Quando mais dados forem acumulados (série ≥ 5 anos semanal), adiciona capacidade neural. |
| 5º | **LSTM/GRU** | Somente com dados em granularidade **semanal** e pelo menos 200+ observações. |

---

### Próximos passos para feature engineering elaborada

1. **Índice de Breteau semanal** por bairro → agregação espacial (kriging ou H3)
2. **Temperatura acumulada** nos últimos 30/60 dias (graus-dia para desenvolvimento larval)
3. **Precipitação acumulada 2–4 semanas** antes (breeding sites)
4. **Variável indicadora de El Niño / ENSO** (SST-Niño3.4)
5. **Desafasagem epidemiológica**: lag entre capturas de armadilha e notificação de caso (~2–4 semanas)
6. **Features de mobilidade urbana** (dados IBGE, viagens carnaval/reveillon)
7. **Série histórica de sorotipos** para capturar ciclos de imunidade da população

---

### Framework de validação recomendado

```
Walk-Forward Validation (Expanding Window):
├── Fold 1: Treino [jan/2020–dez/2022] → Teste [jan/2023]
├── Fold 2: Treino [jan/2020–jan/2023] → Teste [fev/2023]
├── ...
└── Fold N: Treino [jan/2020–mar/2025] → Teste [abr/2025]
```

Evita **data leakage** temporal e estima desempenho real de produção.

---

### Definição epidemiológica de surto (sugestões de limiar)

| Critério | Descrição |
|----------|-----------|
| **Media + σ histórico** | Simples, usado neste notebook |
| **P80 histórico por mês** | Corrige sazonalidade — mais adequado |
| **Critério OMS/PAHO** | Casos > 2× mediana da epidemia de mesmo período nos 5 anos anteriores |
| **Threshold adaptativo** | Calculado por canal endêmico (Cordero-Ruis method) |

In [ ]:
## Resumo final: MAE por modelo e horizonte
modelos_avaliados = {}

for h, df_h in [(1, df_features_h1), (2, df_features_h2), (3, df_features_h3)]:
    X_h = df_h[feature_cols].values
    y_h = df_h["target_casos"].values
    nt  = int(len(X_h) * 0.75)
    m   = RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42, n_jobs=-1)
    m.fit(X_h[:nt], y_h[:nt])
    pr  = np.maximum(m.predict(X_h[nt:]), 0)
    modelos_avaliados[f"RF (h={h})"] = {
        "MAE": round(mean_absolute_error(y_h[nt:], pr), 1),
        "RMSE": round(np.sqrt(mean_squared_error(y_h[nt:], pr)), 1),
    }

modelos_avaliados["SARIMA (h=1)"]       = {"MAE": round(mae_sarima, 1), "RMSE": round(rmse_sarima, 1)}
modelos_avaliados["Holt-Winters (h=1)"] = {"MAE": round(mae_hw, 1),    "RMSE": round(rmse_hw, 1)}
modelos_avaliados["CatBoost (h=1)"]     = {"MAE": round(mae_cb, 1),    "RMSE": round(rmse_cb, 1)}
modelos_avaliados["LightGBM (h=1)"]     = {"MAE": round(mae_lg, 1),    "RMSE": round(rmse_lg, 1)}

df_final = pd.DataFrame(modelos_avaliados).T.sort_values("MAE")

fig, ax = plt.subplots(figsize=(10, 5))
cores = plt.cm.tab10(np.linspace(0, 0.9, len(df_final)))
bars  = ax.barh(df_final.index, df_final["MAE"].astype(float), color=cores, alpha=0.85)
ax.bar_label(bars, fmt="%.1f", padding=3)
ax.set_xlabel("MAE (casos/mês)")
ax.set_title("MAE por Modelo e Horizonte de Previsão")
ax.invert_yaxis()
plt.tight_layout(); plt.show()

print("\nTabela final:")
print(df_final.to_string())